In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [ ]:
!pip install mlflow dagshub
import pandas as pd
import numpy as np
import mlflow
import dagshub
import category_encoders as ce
import gc
import random
import copy
from sklearn.model_selection import train_test_split, ParameterGrid
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler

# Train/Test Split

In [4]:
dagshub_username = "nikaduri"
repo_name = "ml-ieee-cis-fraud-detection"
dagshub.init(repo_owner=dagshub_username, repo_name=repo_name, mlflow=True)

mlflow.set_experiment("RandomForest_Training")

print("Loading data...")
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')

df = train_transaction.merge(train_identity, on='TransactionID', how='left')
del train_transaction, train_identity 
gc.collect()

X = df.drop(columns=['isFraud'])
y = df['isFraud']

X_temp, X_test_raw, y_temp, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

X_train_raw, X_val_raw, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.25, shuffle=False) 

del df, X, y, X_temp, y_temp 
gc.collect()

print(f"Data Splits - Train: {X_train_raw.shape[0]} | Val: {X_val_raw.shape[0]} | Test: {X_test_raw.shape[0]}")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=6fb3c67c-afc4-4862-baa0-76a1b50d39d0&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=004fe56330ec49dd61147394879d3bc9df03122c0a6b4d3f8668bf26ab2ea084




Accessing as nikaduri

Initialized MLflow to track repo "nikaduri/ml-ieee-cis-fraud-detection"

Repository nikaduri/ml-ieee-cis-fraud-detection initialized!

2026/05/03 13:20:37 INFO mlflow.tracking.fluent: Experiment with name 'RandomForest_Training' does not exist. Creating a new experiment.


Loading data...
Data Splits - Train: 354324 | Val: 118108 | Test: 118108


# Feature Cleaning

In [5]:
with mlflow.start_run(run_name="RandomForest_Cleaning"):
    print("\n--- Executing Cleaning Stage ---")
    nan_threshold = 0.90
    mlflow.log_param("nan_drop_threshold", nan_threshold)
    
    irrelevant_cols = ['TransactionID', 'TransactionDT']
    
    missing_fractions = X_train_raw.isnull().mean()
    cols_to_keep_nan = missing_fractions[missing_fractions <= nan_threshold].index.tolist()
    cleaned_features = [c for c in cols_to_keep_nan if c not in irrelevant_cols]
    
    mlflow.log_metric("features_after_cleaning", len(cleaned_features))
    
    del missing_fractions
    gc.collect()


--- Executing Cleaning Stage ---
🏃 View run RandomForest_Cleaning at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/eaebb573675e4a03885f59acb2419f26
🧪 View experiment at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/7


# Feature Selection

In [6]:
with mlflow.start_run(run_name="RandomForest_Feature_Selection"):
    print("\n--- Executing Feature Selection Stage ---")
    corr_threshold = 0.90
    mlflow.log_param("corr_threshold", corr_threshold)
    
    X_train_temp = X_train_raw[cleaned_features].copy()
    cat_cols_temp = X_train_temp.select_dtypes(include=['object']).columns.tolist()
    num_cols_temp = X_train_temp.select_dtypes(exclude=['object']).columns.tolist()
    
    X_train_temp[cat_cols_temp] = X_train_temp[cat_cols_temp].fillna('Missing_Category')
    X_train_temp[num_cols_temp] = X_train_temp[num_cols_temp].fillna(X_train_temp[num_cols_temp].median())
    
    woe = ce.WOEEncoder(cols=cat_cols_temp)
    X_train_encoded = woe.fit_transform(X_train_temp, y_train)
    
    del X_train_temp 
    gc.collect()
    
    corr_matrix = X_train_encoded.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop_corr = [column for column in upper.columns if any(upper[column] > corr_threshold)]
    
    del X_train_encoded, corr_matrix, upper
    gc.collect()
    
    final_features = [c for c in cleaned_features if c not in to_drop_corr]
    categorical_cols = [c for c in final_features if X_train_raw[c].dtype == 'object']
    numerical_cols = [c for c in final_features if X_train_raw[c].dtype != 'object']
    
    mlflow.log_metric("final_feature_count", len(final_features))


--- Executing Feature Selection Stage ---
🏃 View run RandomForest_Feature_Selection at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/c48cb2cfce1243a6b1d6616ae71bc48b
🧪 View experiment at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/7


# Training

In [7]:
with mlflow.start_run(run_name="RandomForest_Training") as parent_run:
    X_train_model = X_train_raw[final_features]
    X_val_model = X_val_raw[final_features]
    X_test_model = X_test_raw[final_features]

    # Preprocessors
    num_transformer = SimpleImputer(strategy='median')
    cat_transformer = ImbPipeline(steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='Missing_Category')),
        ('woe', ce.WOEEncoder())
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_transformer, numerical_cols),
            ('cat', cat_transformer, categorical_cols)
        ],
        remainder='drop' 
    )

    
    pipeline = ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('undersampler', RandomUnderSampler(sampling_strategy=3/7, random_state=67)),
        ('classifier', RandomForestClassifier(class_weight='balanced', random_state=67, n_jobs=-1))
    ])

    param_space = {
        'classifier__n_estimators': [50, 100],
        'classifier__criterion': ['gini'],
        'classifier__max_depth': [8, 12, 16],
        'classifier__min_samples_split': [50, 100],
        'classifier__min_samples_leaf': [25, 50]
    }

    all_combinations = list(ParameterGrid(param_space))
    sampled_combinations = random.sample(all_combinations, min(10, len(all_combinations)))

    best_val_auc = 0
    best_params = None
    best_pipeline = None

    print(f"Starting explicit grid search over {len(sampled_combinations)} combinations...")
    
    for i, params in enumerate(sampled_combinations):
        
        with mlflow.start_run(run_name=f"Manual_Search_Run_{i+1}", nested=True):
            clean_params = {k.replace('classifier__', ''): v for k, v in params.items()}
            mlflow.log_params(clean_params)
            
            pipeline.set_params(**params)
            pipeline.fit(X_train_model, y_train)
            
            y_train_pred = pipeline.predict_proba(X_train_model)[:, 1]
            y_val_pred = pipeline.predict_proba(X_val_model)[:, 1]
            
            train_auc = roc_auc_score(y_train, y_train_pred)
            val_auc = roc_auc_score(y_val, y_val_pred)
            
            mlflow.log_metrics({
                "train_roc_auc": train_auc,
                "val_roc_auc": val_auc
            })
            
            print(f"Run {i+1} | Est: {params['classifier__n_estimators']} | Depth: {params['classifier__max_depth']} | Train AUC: {train_auc:.4f} | Val AUC: {val_auc:.4f}")
            
            if val_auc > best_val_auc:
                best_val_auc = val_auc
                best_params = clean_params
                best_pipeline = copy.deepcopy(pipeline)
                

    y_test_pred = best_pipeline.predict_proba(X_test_model)[:, 1]
    test_auc = roc_auc_score(y_test, y_test_pred)
    
    print(f"Final Test ROC-AUC: {test_auc:.4f}")

    mlflow.log_params({"best_" + k: v for k, v in best_params.items()})
    mlflow.log_metric("final_test_roc_auc", test_auc)
    
    mlflow.sklearn.log_model(
        sk_model=best_pipeline,
        artifact_path="random_forest_pipeline_artifact",
        registered_model_name="RandomForest_Production_Model" 
    )


--- Executing Training & Pipeline Construction ---
Starting explicit grid search over 10 combinations...
Run 1 | Est: 50 | Depth: 8 | Train AUC: 0.8706 | Val AUC: 0.8697
🏃 View run Manual_Search_Run_1 at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/516520aaf0534851bae35f02eca9ae4d
🧪 View experiment at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/7
Run 2 | Est: 50 | Depth: 12 | Train AUC: 0.8823 | Val AUC: 0.8782
🏃 View run Manual_Search_Run_2 at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/ed31229dfbf140bab58b9a462aa71ef8
🧪 View experiment at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/7
Run 3 | Est: 100 | Depth: 16 | Train AUC: 0.8918 | Val AUC: 0.8823
🏃 View run Manual_Search_Run_3 at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/c5fca51f05144016b0dcf360c045520b
🧪 View experiment at: https://dag

2026/05/03 13:29:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 13:29:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'RandomForest_Production_Model'.
2026/05/03 13:29:52 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: RandomForest_Production_Model, version 1
Created version '1' of model 'RandomForest_Production_Model'.


🏃 View run RandomForest_Training at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/7/runs/b67da908a73d4ab4bda34ad059024a05
🧪 View experiment at: https://dagshub.com/nikaduri/ml-ieee-cis-fraud-detection.mlflow/#/experiments/7
